In [1]:
import pandas as pd
import sklearn as sk
import numpy as np

<h1>1.0 Import Ride and Zone Lookup Data</h1>

<h2>1.1 Import Ride Data</h2>

In [2]:
rides = pd.read_csv(
    "../dataset/rides_train_sample.csv",
    dtype=dict(
        VendorID=int,
        tpep_pickup_datetime=str,
        tpep_dropoff_datetime=str,
        passenger_count=float,
        trip_distance=float,
        RatecodeID=float,
        store_and_fwd_flag=str,
        PULocationID=int,
        DOLocationID=int,
        payment_type=int,
        fare_amount=float,
        extra=float,
        mta_tax=float,
        tip_amount=float,
        tolls_amount=float,
        improvement_surcharge=float,
        total_amount=float,
        congestion_surcharge=float,
        Airport_fee=float,
    ),
)

<h2>1.2 Import Zone Lookup Data</h2>

In [3]:
# Find Manhattan zone IDs for filtering up next
zones = pd.read_excel('../dataset/taxi_zone_lookup.xlsx')
manhat_zones = zones[zones['Borough'] == 'Manhattan']['LocationID']

<h2>1.3 Check Data Imports</h2>

In [4]:
# rides.shape
# rides.info()
rides.describe()
#rides.head()

# zones.info()
#manhat_zones.head()

,Unnamed: 0,VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
count,2.000000e+06,2.000000e+06,1.807412e+06,2.000000e+06,1.807412e+06,2.000000e+06,2.000000e+06,2.000000e+06,2.000000e+06,2.000000e+06,2.000000e+06,2.000000e+06,2.000000e+06,2.000000e+06,2.000000e+06,1.807412e+06,1.807412e+06
mean,1.678174e+06,1.764142e+00,1.336039e+00,4.827009e+00,2.331517e+00,1.641172e+02,1.634468e+02,1.112453e+00,1.946138e+01,1.390647e+00,4.797863e-01,3.316701e+00,5.639295e-01,9.629970e-01,2.804226e+01,2.230589e+00,1.485680e-01
std,1.111724e+06,4.259975e-01,8.174255e-01,3.543436e+02,1.096513e+01,6.430717e+01,6.964609e+01,6.503011e-01,2.407163e+02,1.818526e+00,1.301921e-01,4.085779e+00,2.227406e+00,2.549519e-01,2.411417e+02,8.756406e-01,5.044024e-01
min,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,-9.990000e+02,-7.500000e+00,-5.000000e-01,-9.901000e+01,-1.167000e+02,-1.000000e+00,-1.000000e+03,-2.500000e+00,-1.750000e+00
25%,6.509465e+05,2.000000e+00,1.000000e+00,1.010000e+00,1.000000e+00,1.320000e+02,1.130000e+02,1.000000e+00,9.300000e+00,0.000000e+00,5.000000e-01,0.000000e+00,0.000000e+00,1.000000e+00,1.575000e+01,2.500000e+00,0.000000e+00
50%,1.396293e+06,2.000000e+00,1.000000e+00,1.760000e+00,1.000000e+00,1.610000e+02,1.620000e+02,1.000000e+00,1.350000e+01,1.000000e+00,5.000000e-01,2.600000e+00,0.000000e+00,1.000000e+00,2.100000e+01,2.500000e+00,0.000000e+00
75%,2.688128e+06,2.000000e+00,1.000000e+00,3.370000e+00,1.000000e+00,2.330000e+02,2.340000e+02,1.000000e+00,2.260000e+01,2.500000e+00,5.000000e-01,4.250000e+00,0.000000e+00,1.000000e+00,3.060000e+01,2.500000e+00,0.000000e+00
max,3.833770e+06,7.000000e+00,9.000000e+00,2.232228e+05,9.900000e+01,2.650000e+02,2.650000e+02,4.000000e+00,3.355444e+05,1.444000e+01,1.050000e+01,2.156600e+02,1.071000e+02,1.000000e+00,3.355509e+05,2.500000e+00,1.750000e+00


<h1>2.0 Filtering and Exploring the Data</h1>

<h2>2.1 Filter for Manhattan Pickup Zones</h2>

In [5]:
manhat_rides = rides[rides['PULocationID'].isin(manhat_zones)].copy()

<h2>2.2 Check the filtered dataframe</h2>

In [6]:
# manhat_rides.shape
# manhat_rides.info()
manhat_rides.describe()

,Unnamed: 0,VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
count,1.766579e+06,1.766579e+06,1.596157e+06,1.766579e+06,1.596157e+06,1.766579e+06,1.766579e+06,1.766579e+06,1.766579e+06,1.766579e+06,1.766579e+06,1.766579e+06,1.766579e+06,1.766579e+06,1.766579e+06,1.596157e+06,1.596157e+06
mean,1.677737e+06,1.764988e+00,1.329723e+00,3.726164e+00,1.468517e+00,1.681171e+02,1.651059e+02,1.100885e+00,1.593618e+01,1.239918e+00,4.831154e-01,2.827637e+00,2.354296e-01,9.711522e-01,2.355578e+01,2.366292e+00,1.118311e-04
std,1.110320e+06,4.245200e-01,8.160460e-01,3.525203e+02,6.476389e+00,6.531588e+01,6.865372e+01,6.327816e-01,2.556312e+02,1.506922e+00,1.212323e-01,3.058653e+00,1.592578e+00,2.324736e-01,2.558104e+02,6.992403e-01,1.567091e-02
min,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,4.000000e+00,1.000000e+00,0.000000e+00,-7.485000e+02,-7.500000e+00,-5.000000e-01,-9.901000e+01,-4.600000e+01,-1.000000e+00,-7.633800e+02,-2.500000e+00,-1.750000e+00
25%,6.530015e+05,2.000000e+00,1.000000e+00,9.800000e-01,1.000000e+00,1.370000e+02,1.140000e+02,1.000000e+00,8.600000e+00,0.000000e+00,5.000000e-01,0.000000e+00,0.000000e+00,1.000000e+00,1.540000e+01,2.500000e+00,0.000000e+00
50%,1.393026e+06,2.000000e+00,1.000000e+00,1.600000e+00,1.000000e+00,1.630000e+02,1.620000e+02,1.000000e+00,1.280000e+01,1.000000e+00,5.000000e-01,2.560000e+00,0.000000e+00,1.000000e+00,1.990000e+01,2.500000e+00,0.000000e+00
75%,2.687831e+06,2.000000e+00,1.000000e+00,2.680000e+00,1.000000e+00,2.340000e+02,2.340000e+02,1.000000e+00,1.910000e+01,2.500000e+00,5.000000e-01,3.980000e+00,0.000000e+00,1.000000e+00,2.690000e+01,2.500000e+00,0.000000e+00
max,3.833770e+06,7.000000e+00,9.000000e+00,2.232228e+05,9.900000e+01,2.630000e+02,2.650000e+02,4.000000e+00,3.355444e+05,1.444000e+01,1.050000e+01,2.156600e+02,9.999000e+01,1.000000e+00,3.355509e+05,2.500000e+00,1.750000e+00


<h2>2.3 Check for missing data in general</h2>

In [7]:
print(manhat_rides.isna().mean() * 100)

Unnamed: 0               0.000000
VendorID                 0.000000
tpep_pickup_datetime     0.000000
tpep_dropoff_datetime    0.000000
passenger_count          9.647007
trip_distance            0.000000
RatecodeID               9.647007
store_and_fwd_flag       9.647007
PULocationID             0.000000
DOLocationID             0.000000
payment_type             0.000000
fare_amount              0.000000
extra                    0.000000
mta_tax                  0.000000
tip_amount               0.000000
tolls_amount             0.000000
improvement_surcharge    0.000000
total_amount             0.000000
congestion_surcharge     9.647007
Airport_fee              9.647007
dtype: float64


<h2>2.4 Check for missing data by vendor</h2>

In [8]:
missing_by_vendor = (
    manhat_rides
    .isna()
    .groupby(manhat_rides['VendorID'])
    .mean() * 100
).round(5)

missing_by_vendor = missing_by_vendor.loc[:, (missing_by_vendor > 0).any()]

print(missing_by_vendor)

          passenger_count  RatecodeID  store_and_fwd_flag  \
VendorID                                                    
1                 9.07486     9.07486             9.07486   
2                 9.82125     9.82125             9.82125   
6               100.00000   100.00000           100.00000   
7                 0.00000     0.00000             0.00000   

          congestion_surcharge  Airport_fee  
VendorID                                     
1                      9.07486      9.07486  
2                      9.82125      9.82125  
6                    100.00000    100.00000  
7                      0.00000      0.00000  


<h2>2.5 Updating data types</h2>

In [9]:
# Convert to datetime
manhat_rides['tpep_pickup_datetime'] = pd.to_datetime(
    manhat_rides['tpep_pickup_datetime'],
    errors='coerce'
)

manhat_rides['tpep_dropoff_datetime'] = pd.to_datetime(
    manhat_rides['tpep_dropoff_datetime'],
    errors='coerce'
)

# Convert Y/N to boolean
manhat_rides['store_and_fwd_flag'] = (
    manhat_rides['store_and_fwd_flag']
    .map({'Y': True, 'N': False})
    .astype('boolean') # just for reference clarity
)

<h2>2.6 Check data types</h2>

In [10]:
manhat_rides.dtypes

Unnamed: 0                        int64
VendorID                          int64
tpep_pickup_datetime     datetime64[ns]
tpep_dropoff_datetime    datetime64[ns]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag              boolean
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
dtype: object

<h2>2.7 Preliminary feature evaluation</h2>

| Feature                  | Type        | Missing Data | Probably Important | Notes |
|--------------------------|------------|------|------|------|
| VendorID                | Categorical | No | No |  |
| tpep_pickup_datetime    | Datetime    | No | Yes | Needs conversion |
| tpep_dropoff_datetime   | Datetime    | No | Yes | Needs conversion |
| passenger_count         | Discrete    | Yes | No | Leakage risk|
| trip_distance           | Continuous  | No | No | Leakage risk|
| RatecodeID              | Categorical | Yes | Yes |  |
| store_and_fwd_flag      | Boolean     | Yes | No | Apparently useless |
| PULocationID            | Categorical | No | Yes |  |
| DOLocationID            | Categorical | No | Yes |  |
| payment_type            | Categorical | No | No |  |
| fare_amount             | Continuous  | No | No | Part of revenue |
| extra                   | Continuous  | No | No | Part of revenue |
| mta_tax                 | Continuous  | No | No | Part of revenue |
| tip_amount              | Continuous  | No | No | Part of revenue |
| tolls_amount            | Continuous  | No | No | Part of revenue |
| improvement_surcharge   | Continuous  | No | No | Part of revenue |
| total_amount            | Continuous  | No | Yes |  Target variable (revenue) |
| congestion_surcharge    | Continuous  | Yes | No | Part of revenue |
| airport_fee             | Continuous  | Yes | No | Part of revenue |

<h1>3.0 Finding Data Discrepancies and Cleaning</h1>

<h2>3.1 Add a duration and avg speed column for validation</h2>

In [11]:
# Duration in hours
manhat_rides['duration_hours'] = (
    (manhat_rides['tpep_dropoff_datetime'] - manhat_rides['tpep_pickup_datetime'])
    .dt.total_seconds() / 3600
)

# Average speed (mph)
manhat_rides['avg_speed_mph'] = (
    manhat_rides['trip_distance'] /
    manhat_rides['duration_hours'].replace(0, np.nan)
)

<h2>3.1 Finding unphysical data</h2>

In [12]:
def get_constraint_violations(df):

    violations = {}
    # These are the cut-offs we apply to the data. They are decided and refined using statistical assessment in 3.2
    # The conditions describe what data will be flagged
    possible_checks = {
        'passenger_count': lambda x: x < 0,
        'trip_distance': lambda x: (x <= 0) | (x > 300), # to ensure the taxi actually moved
        'fare_amount': lambda x: (x < 0) | (x > 3000),
        'extra': lambda x: x < 0,
        'mta_tax': lambda x: x < 0,
        'tip_amount': lambda x: x < 0,
        'tolls_amount': lambda x: x < 0,
        'improvement_surcharge': lambda x: x < 0,
        'total_amount': lambda x: x < 0,
        'congestion_surcharge': lambda x: x < 0,
        'Airport_fee': lambda x: x < 0,
        'duration_hours': lambda x: (x <= 0) | (x > 12), # to remove any entries which took zero time
        'avg_speed_mph': lambda x: (x < 0.1) | (x > 150) # to remove any entries which suggest the taxi has moved unrealistically far in a given time.
    }

    # Column-based checks
    for col, condition in possible_checks.items():
        if col in df.columns:
            mask = condition(df[col])
            violations[col] = df.index[mask].tolist()

    # Datetime check
    if 'tpep_pickup_datetime' in df.columns and 'tpep_dropoff_datetime' in df.columns:
        mask = df['tpep_dropoff_datetime'] < df['tpep_pickup_datetime']
        violations['pickup_before_dropoff'] = df.index[mask].tolist()

    # Combine all violating indices
    all_bad_indices = set()
    for v in violations.values():
        all_bad_indices.update(v)

    return violations, all_bad_indices

In [13]:
# Get the violation df and the row numbers of offending data entries
violations, bad_idx = get_constraint_violations(manhat_rides)

# Print a summary in order to check how much is flagged. We want this to be relatively small compared to sample size.
print({k: len(v) for k, v in violations.items()})
print(f"Total bad rows: {len(bad_idx)}")

{'passenger_count': 0, 'trip_distance': 27284, 'fare_amount': 28465, 'extra': 11259, 'mta_tax': 22424, 'tip_amount': 25, 'tolls_amount': 711, 'improvement_surcharge': 23009, 'total_amount': 23117, 'congestion_surcharge': 22033, 'Airport_fee': 13, 'duration_hours': 1334, 'avg_speed_mph': 28752, 'pickup_before_dropoff': 91}
Total bad rows: 56293


In [14]:
#Inspect the bad rows in case of suspicion
manhat_rides.loc[list(bad_idx)].head()

,Unnamed: 0,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,duration_hours,avg_speed_mph
1703936,2953897,1,2024-01-28 14:24:25,2024-01-28 14:30:57,NaN,0.00,NaN,<NA>,236,141,...,0.0,0.5,0.00,0.0,1.0,10.83,NaN,NaN,0.108889,0.000000
1572865,411300,1,2024-08-05 14:14:52,2024-08-05 14:19:33,1.0,0.00,1.0,False,140,263,...,2.5,0.5,1.95,0.0,1.0,11.75,2.5,0.0,0.078056,0.000000
131073,3429006,2,2024-03-22 18:41:14,2024-03-22 18:50:36,NaN,2.39,NaN,<NA>,140,170,...,0.0,0.5,0.00,0.0,1.0,2.39,NaN,NaN,0.156111,15.309609
655360,1445638,2,2024-12-12 21:46:05,2024-12-12 22:00:48,1.0,1.39,1.0,False,142,229,...,-1.0,-0.5,0.00,0.0,-1.0,-18.50,-2.5,0.0,0.245278,5.667044
1179659,228138,2,2024-03-03 03:52:38,2024-03-03 04:00:10,1.0,0.00,1.0,False,148,144,...,1.0,0.5,2.58,0.0,1.0,15.48,2.5,0.0,0.125556,0.000000


<h2>3.2 Removing unphysical data</h2>

In [15]:
# Remove the data rows that were identified as having constraint issues
manhat_rides_clean = manhat_rides.drop(index=bad_idx).copy()

#Quick check
print(manhat_rides.shape)
print(manhat_rides_clean.shape)

(1766579, 22)
(1710286, 22)


<h2>3.3 Check for statistically suspicious data (to validate cut-off choices)</h2>

In [17]:
# Have a look at statistics
manhat_rides_clean.describe().loc[['mean', 'std', 'max', 'min']]

,Unnamed: 0,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,duration_hours,avg_speed_mph
mean,1.664197e+06,1.763362,2024-07-04 04:46:10.468608768,2024-07-04 05:00:42.009005824,1.328730,2.419629,1.423298,168.297459,165.222852,1.076927,...,1.281946,0.497174,2.885509,0.242004,0.997410,23.902615,2.440590,0.000095,0.242095,9.737096
std,1.104861e+06,0.425197,NaN,NaN,0.815527,2.750469,6.226179,65.233914,68.615006,0.556979,...,1.494524,0.040054,3.010705,1.578583,0.050641,14.932136,0.380782,0.012927,0.179460,4.885397
max,3.833770e+06,6.000000,2024-12-31 23:58:49,2025-01-01 00:19:33,9.000000,187.910000,99.000000,263.000000,265.000000,4.000000,...,14.440000,10.500000,215.660000,99.990000,1.000000,811.220000,2.500000,1.750000,11.918611,150.000000
min,0.000000e+00,1.000000,2002-12-31 22:10:04,2002-12-31 22:14:22,0.000000,0.010000,1.000000,4.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000278,0.100699


In [18]:
# Check toll amounts manually as they seem a bit high. All checks out.
print(manhat_rides_clean[manhat_rides_clean['tolls_amount'] > 50]['tolls_amount'])

294610     59.63
340199     69.13
743786     98.00
758344     50.76
823472     63.52
833043     80.00
886271     67.00
1050210    55.38
1362423    93.38
1727020    99.99
1845970    69.40
Name: tolls_amount, dtype: float64


In [19]:
import numpy as np
import pandas as pd

# Numerical features to be tested (judged using above describe() print and particularly maxima
# All maxima are satisfactory. 'tolls_amount' needed extra analysis, but all seems in order. We now Z-test the remaining features.
# Note fare_amount <= total_amount
iqrtest_features = [
    'trip_distance',
    'total_amount',
]

def get_iqr_violations(df, features, k=1.5):

    # Store flagged row indices separately for each feature
    violations = {}
    # Store flagged row indices separately for each feature
    flagged_rows = []
    # Store all unique bad row indices across all tested features
    bad_idx = set()

    # Loop through each feature that's selected
    for col in features:
        # Skip if not in the dataframe
        if col not in df.columns:
            continue

        # If the column has no non-missing values, record no violations
        x = df[col].dropna()
        if x.empty:
            violations[col] = []
            continue

        # Quantiles
        q1 = x.quantile(0.25)
        q3 = x.quantile(0.75)
        # Interquantile range (IQR)
        iqr = q3 - q1

       # Thresholds
        lower = q1 - k * iqr
        upper = q3 + k * iqr

        # Flag rows lying outside acceptable range
        mask = (df[col] < lower) | (df[col] > upper)
        idx_list = df.index[mask.fillna(False)].tolist()

        # Store violating row numbers
        violations[col] = idx_list
        # Add bad row numbers to the set
        bad_idx.update(idx_list)

        # Store some info on each flagged row (for review)
        for idx in idx_list:
            flagged_rows.append({
                'row_index': idx,
                'feature': col,
                'value': df.at[idx, col]
            })

    flagged_df = pd.DataFrame(flagged_rows)

    return violations, flagged_df, bad_idx

In [20]:
violations_iqr, flagged_iqr_df, bad_idx_iqr = get_iqr_violations(
    manhat_rides_clean,
    features=iqrtest_features,
    k=5 # adjust for sensitivity
)

In [21]:
flagged_iqr_df.sort_values(by='value')

,row_index,feature,value
33070,1920385,trip_distance,11.21
16794,973377,trip_distance,11.21
25487,1475339,trip_distance,11.21
18515,1074738,trip_distance,11.21
8019,461903,trip_distance,11.21
...,...,...,...
47174,1105900,total_amount,506.90
50864,1429535,total_amount,534.38
38275,333424,total_amount,549.66
52941,1613055,total_amount,556.28


In [22]:
# Have a look at the number of flagged entries in each selected feature.
print({k: len(v) for k, v in violations_iqr.items()})
print(f"Total unique bad rows: {len(bad_idx_iqr)}")

{'trip_distance': 34434, 'total_amount': 22992}
Total unique bad rows: 36569


In [23]:
# Inspect original rows (very many)
manhat_rides_clean[['duration_hours', 'avg_speed_mph', 'trip_distance', 'total_amount', 'fare_amount', 'PULocationID', 'DOLocationID']].loc[list(bad_idx_iqr)].sort_values(by='total_amount', ascending=False)

,duration_hours,avg_speed_mph,trip_distance,total_amount,fare_amount,PULocationID,DOLocationID
1194016,3.684167,30.742366,113.26,811.22,777.90,88,265
1613055,5.490833,34.222492,187.91,556.28,555.28,144,265
333424,1.682778,42.138660,70.91,549.66,515.40,233,265
1429535,2.551389,61.335220,156.49,534.38,500.00,48,265
1105900,1.370833,57.279027,78.52,506.90,504.90,161,265
...,...,...,...,...,...,...,...
330378,0.725278,25.645347,18.60,1.01,0.01,107,170
1974455,0.819167,20.264496,16.60,1.01,0.01,100,132
285649,0.774444,21.305595,16.50,1.00,0.00,230,132
1716293,0.936111,19.014837,17.80,1.00,0.00,230,132
